# Session 05: Image Stitching & Homographies

**CVI4IC — Summer Semester 2026**

Topics: local features for alignment, translational vs. projective models, robust estimation (RANSAC), `warpAffine` / `warpPerspective`, simple blending, and OpenCV’s `Stitcher`.

## Table of contents

1. Overlapping crops ("campus") and ground-truth translation
2. SIFT matching + Lowe ratio
3. Geometric interpretation ($\Delta x$, $\Delta y$)
4. Least-squares translation
5. Outliers, least squares vs. RANSAC
6. Homography for two natural views (`foto1A` / `foto1B`)
7. 2D transformation families (translation, affine, perspective)
8. Forward vs. inverse warping + interpolation
9. Warping, canvas, alpha blending
10. Translational merge on campus crops
11. OpenCV `Stitcher`
12. ✏️ Exercises

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy scipy

import os
import urllib.request

import cv2
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import least_squares

In [ ]:
def download_file(url: str, path: str) -> None:
    if os.path.isfile(path):
        return
    urllib.request.urlretrieve(url, path)


def to_rgb(image_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def imshow_bgr(image_bgr: np.ndarray, *, title=None, **kwargs) -> None:
    plt.imshow(to_rgb(image_bgr), **kwargs)
    plt.axis("off")
    if title:
        plt.title(title)


BASE = "https://raw.githubusercontent.com/Digital-Media/cv_data/main/panorama_stitching"
download_file(f"{BASE}/campus_hagenberg.jpg", "campus.jpg")
download_file(f"{BASE}/UTA_foto1A.jpg", "foto1A.jpg")
download_file(f"{BASE}/UTA_foto1B.jpg", "foto1B.jpg")

print("Data ready.")

## 1. Overlapping crops — when motion is *almost* a translation

We cut two windows from one high-resolution image. The overlap region is consistent with a **2D translation** between the crops (ground truth used in the checks below: **t ≈ (449, −41)** in pixel units).

In [ ]:
import math

GROUND_TRUTH_T = np.array([449.0, -41.0], dtype=np.float32)

huge = cv2.imread("campus.jpg")
w = 1250
h = math.floor(w * huge.shape[0] / huge.shape[1])
img = cv2.resize(huge, (w, h))

img_a = img[:400, -801:-1, :].copy()
img_b = img[-401:-1, :800, :].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(to_rgb(img_a))
axes[0].set_title("Crop A (top-right region)")
axes[0].axis("off")
axes[1].imshow(to_rgb(img_b))
axes[1].set_title("Crop B (bottom-left region)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("img_a:", img_a.shape, "img_b:", img_b.shape)

## 2. SIFT matching + Lowe’s ratio test

We reuse the same ingredients as in Session 04: SIFT descriptors, k-NN matching, and **Lowe’s ratio test** to reject ambiguous matches. A **lower** ratio is stricter (fewer matches, usually cleaner).

In [ ]:
def get_matched_points(kps_a, kps_b, matches):
    # Matches come from knnMatch(des_b, des_a): query -> B, train -> A
    pts_a = np.float32([kps_a[m.trainIdx].pt for m in matches])
    pts_b = np.float32([kps_b[m.queryIdx].pt for m in matches])
    return pts_a, pts_b


def ratio_test_matches(des_a, des_b, ratio: float):
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    pairs = bf.knnMatch(des_b, des_a, k=2)
    good = []
    for pair in pairs:
        if len(pair) < 2:
            continue
        m, n = pair
        if m.distance < ratio * n.distance:
            good.append(m)
    return sorted(good, key=lambda m: m.distance)


def sift_match_visual(im_a_bgr, im_b_bgr, ratio: float = 0.5, max_draw: int = 60, show: bool = True):
    gray_a = cv2.cvtColor(im_a_bgr, cv2.COLOR_BGR2GRAY)
    gray_b = cv2.cvtColor(im_b_bgr, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create()
    kps_a, des_a = sift.detectAndCompute(gray_a, None)
    kps_b, des_b = sift.detectAndCompute(gray_b, None)
    matches = ratio_test_matches(des_a, des_b, ratio)
    pts_a, pts_b = get_matched_points(kps_a, kps_b, matches)

    if show and max_draw > 0 and len(matches) > 0:
        n = min(max_draw, len(matches))
        vis = cv2.drawMatches(
            gray_b,
            kps_b,
            gray_a,
            kps_a,
            matches[:n],
            None,
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
        )
        plt.figure(figsize=(18, 8))
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title(f"SIFT matches (Lowe ratio = {ratio}), showing {n} of {len(matches)}")
        plt.tight_layout()
        plt.show()

    return pts_a, pts_b, matches


pts_a, pts_b, matches = sift_match_visual(img_a, img_b, ratio=0.5)

## 3. Geometric structure of the matches

### 3.1 Offsets $\Delta x, \Delta y$

For each correspondence, plot **offset** $\mathbf{x}_A - \mathbf{x}_B$. Inliers should form a tight cluster when the motion is a pure translation.

In [ ]:
diff = pts_a - pts_b
distances = np.float32([m.distance for m in matches])

plt.figure(figsize=(8, 8))
plt.scatter(diff[:, 0], diff[:, 1], c=distances, cmap="viridis", s=12)
plt.colorbar(label="descriptor distance")
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel(r"$\Delta x$")
plt.ylabel(r"$\Delta y$")
plt.title("Offsets for each SIFT match (campus crops)")
plt.tight_layout()
plt.show()

### 3.2 Coordinate-wise plots

If $\mathbf{x}' = \mathbf{x} + \mathbf{t}$, then $x'_x \approx x_x + t_x$ (and similarly for $y$). The red line shows the least-squares fit after Section 4.

## 4. Least squares translation

We model $\mathbf{x}_A \approx \mathbf{x}_B + \mathbf{t}$ and minimize $\sum_i \|\mathbf{x}_{A,i} - \mathbf{x}_{B,i} - \mathbf{t}\|_2^2$ by letting `scipy` optimize $\mathbf{t}$. Residuals are stacked **per coordinate** (not one norm per point), which matches ordinary least squares for Gaussian noise.

In [ ]:
def fit_translation_ls(pts_a_xy: np.ndarray, pts_b_xy: np.ndarray):
    def residuals(t_vec):
        return (pts_a_xy - (pts_b_xy + t_vec)).ravel()

    t0 = np.zeros(2, dtype=np.float64)
    sol = least_squares(residuals, t0)
    return sol.x.astype(np.float32)


t_hat = fit_translation_ls(pts_a, pts_b)
print("Estimated t:", t_hat)
print("Ground truth t:", GROUND_TRUTH_T)
print("L2 error (pixels):", float(np.linalg.norm(t_hat - GROUND_TRUTH_T)))

model_b = pts_a - t_hat.reshape(1, 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(pts_a[:, 0], pts_b[:, 0], s=10, alpha=0.6)
axes[0].plot(pts_a[:, 0], model_b[:, 0], color="red", linewidth=1)
axes[0].set_xlabel(r"$x$ in A")
axes[0].set_ylabel(r"$x$ in B")
axes[0].set_title("Horizontal coordinates")
axes[1].scatter(pts_a[:, 1], pts_b[:, 1], s=10, alpha=0.6)
axes[1].plot(pts_a[:, 1], model_b[:, 1], color="red", linewidth=1)
axes[1].set_xlabel(r"$y$ in A")
axes[1].set_ylabel(r"$y$ in B")
axes[1].set_title("Vertical coordinates")
plt.tight_layout()
plt.show()

## 5. Outliers and RANSAC

Relax the ratio test (e.g. **0.95**) to deliberately admit bad matches. Ordinary least squares is **not robust**: a handful of outliers can drag $\hat{\mathbf{t}}$ far away from the ground truth.

**RANSAC** repeatedly samples minimal sets, fits a model, and keeps the hypothesis with the largest **consensus set**. OpenCV exposes this via `estimateAffine2D(..., method=cv2.RANSAC)` (translation-only is a special case of an affine map).

In [ ]:
pts_a_noisy, pts_b_noisy, matches_noisy = sift_match_visual(img_a, img_b, ratio=0.95, max_draw=80)

t_bad = fit_translation_ls(pts_a_noisy, pts_b_noisy)
print("LS estimate with noisy matches:", t_bad)
print("Ground truth t:", GROUND_TRUTH_T)

affine, inliers = cv2.estimateAffine2D(pts_b_noisy, pts_a_noisy, method=cv2.RANSAC)
if affine is None:
    print("Affine estimation failed.")
else:
    t_ransac = affine[:, 2].astype(np.float32)
    print("RANSAC translation:", t_ransac)
    print("Inliers:", int(inliers.sum()), "/", len(inliers))

## 6. Homography between two views (`foto1A` / `foto1B`)

For a **planar** scene or **pure camera rotation** about the optical center, image-to-image motion is well approximated by a **homography** $\mathbf{x}' \sim \mathbf{H}\mathbf{x}$ (homogeneous coordinates).

We estimate $\mathbf{H}$ with `cv2.findHomography` using **RANSAC** to reject outlier matches.

In [ ]:
im1 = cv2.imread("foto1A.jpg")
im2 = cv2.imread("foto1B.jpg")
assert im1 is not None and im2 is not None

g1 = cv2.cvtColor(im1, cv2.COLOR_BGR2GRAY)
g2 = cv2.cvtColor(im2, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
k1, d1 = sift.detectAndCompute(g1, None)
k2, d2 = sift.detectAndCompute(g2, None)

bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
raw = bf.knnMatch(d1, d2, k=2)
good = [m for m, n in raw if m.distance < 0.75 * n.distance]

pts1 = np.float32([k1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
pts2 = np.float32([k2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

H, mask_h = cv2.findHomography(pts1, pts2, cv2.RANSAC, 5.0)
print("Homography H (3x3):\n", H)
print("Inlier matches:", int(mask_h.sum()), "/", len(good))

inlier_matches = [m for m, keep in zip(good, mask_h.ravel().astype(bool)) if keep]
vis = cv2.drawMatches(im1, k1, im2, k2, inlier_matches[:60], None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(18, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("RANSAC inliers used for H")
plt.tight_layout()
plt.show()

## 7. 2D transformation families (slide recap)

From the lecture: image alignment can be modeled with increasing flexibility:

- **Translation**: $\mathbf{x}' = \mathbf{x} + \mathbf{t}$ (2 params)
- **Rigid / Euclidean**: rotation + translation (3 params)
- **Similarity**: scale + rotation + translation (4 params)
- **Affine**: $\mathbf{x}' = A\mathbf{x} + \mathbf{t}$ (6 params)
- **Perspective / Homography**: $\mathbf{x}' \sim H\mathbf{x}$ (8 DoF)

Rule of thumb: start with the simplest model that explains your data; use homography for planar scenes or pure camera rotation.

## 8. Forward vs. inverse warping + interpolation

Slide takeaway: practical implementations use **inverse warping**.

- **Forward warping** maps each source pixel to destination (can create holes).
- **Inverse warping** samples each destination pixel from source via inverse transform.
- Interpolation matters: nearest, bilinear, bicubic (`cv2.INTER_NEAREST`, `cv2.INTER_LINEAR`, `cv2.INTER_CUBIC`).

OpenCV `warpAffine` / `warpPerspective` follow this inverse-mapping idea internally.

In [ ]:
# Interpolation demo for a translation warp
H_demo = np.float32([[1, 0, 80], [0, 1, -30]])
out_size = (img_a.shape[1] + 200, img_a.shape[0] + 120)

warp_nn = cv2.warpAffine(img_a, H_demo, out_size, flags=cv2.INTER_NEAREST)
warp_lin = cv2.warpAffine(img_a, H_demo, out_size, flags=cv2.INTER_LINEAR)
warp_cub = cv2.warpAffine(img_a, H_demo, out_size, flags=cv2.INTER_CUBIC)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(to_rgb(warp_nn))
axes[0].set_title("Nearest")
axes[1].imshow(to_rgb(warp_lin))
axes[1].set_title("Bilinear")
axes[2].imshow(to_rgb(warp_cub))
axes[2].set_title("Bicubic")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Warping, canvas, and simple blending

`warpPerspective` applies the **inverse** map internally: for each destination pixel it samples the source using $\mathbf{H}^{-1}$. We compute a canvas large enough for both images after warping `foto1B` into the coordinate system of `foto1A`, then blend with normalized masks (alpha feathering in the overlap).

In [ ]:
def stitch_pair_homography(im_left_bgr: np.ndarray, im_right_bgr: np.ndarray, H_right_to_left: np.ndarray):
    """Warp RIGHT image into LEFT's plane; H maps homogeneous coords from right -> left."""
    h_l, w_l = im_left_bgr.shape[:2]
    h_r, w_r = im_right_bgr.shape[:2]

    corners_r = np.float32([[[0, 0]], [[w_r, 0]], [[w_r, h_r]], [[0, h_r]]])
    warped_r_corners = cv2.perspectiveTransform(corners_r, H_right_to_left)
    corners_l = np.float32([[[0, 0]], [[w_l, 0]], [[w_l, h_l]], [[0, h_l]]])
    pts = np.concatenate([corners_l, warped_r_corners], axis=0)
    xmin = int(np.floor(pts[:, 0, 0].min()))
    ymin = int(np.floor(pts[:, 0, 1].min()))
    xmax = int(np.ceil(pts[:, 0, 0].max()))
    ymax = int(np.ceil(pts[:, 0, 1].max()))

    out_w = xmax - xmin
    out_h = ymax - ymin
    T = np.array([[1, 0, -xmin], [0, 1, -ymin], [0, 0, 1]], dtype=np.float64)

    warp_l = cv2.warpPerspective(im_left_bgr, T, (out_w, out_h))
    warp_r = cv2.warpPerspective(im_right_bgr, T @ H_right_to_left, (out_w, out_h))

    mask_l = (cv2.cvtColor(warp_l, cv2.COLOR_BGR2GRAY) > 0).astype(np.float32)
    mask_r = (cv2.cvtColor(warp_r, cv2.COLOR_BGR2GRAY) > 0).astype(np.float32)
    eps = 1e-6
    alpha = mask_l / (mask_l + mask_r + eps)
    alpha3 = np.stack([alpha, alpha, alpha], axis=2)

    blended = (alpha3 * warp_l.astype(np.float32) + (1.0 - alpha3) * warp_r.astype(np.float32)).astype(np.uint8)
    return blended, warp_l, warp_r


H_2to1, _ = cv2.findHomography(pts2, pts1, cv2.RANSAC, 5.0)
panorama, wl, wr = stitch_pair_homography(im1, im2, H_2to1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(to_rgb(wl))
axes[0].set_title("Warped left")
axes[0].axis("off")
axes[1].imshow(to_rgb(wr))
axes[1].set_title("Warped right")
axes[1].axis("off")
axes[2].imshow(to_rgb(panorama))
axes[2].set_title("Alpha blend")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 10. Translational merge on the campus crops

For the synthetic campus pair, warping with an **affine** translation matches the lecture story: align A onto B using $\hat{\mathbf{t}}$ from RANSAC or least squares, then paste B underneath (same idea as the classic VCO alignment notebook).

In [ ]:
pts_a_c, pts_b_c, _ = sift_match_visual(img_a, img_b, ratio=0.5, show=False)
Afit, inl = cv2.estimateAffine2D(pts_b_c, pts_a_c, method=cv2.RANSAC)
t_use = Afit[:, 2].astype(np.float32)

H_tr = np.float32([[1, 0, float(t_use[0])], [0, 1, float(t_use[1])]])
width = img_a.shape[1] + img_b.shape[1] // 2
height = img_a.shape[0] + img_b.shape[0] // 3
canvas = cv2.warpAffine(img_a, H_tr, (width, height))
canvas[0 : img_b.shape[0], 0 : img_b.shape[1]] = img_b

plt.figure(figsize=(16, 8))
imshow_bgr(canvas, title="Campus crops: translate A, overlay B (naive paste)")
plt.tight_layout()
plt.show()

## 11. OpenCV `Stitcher`

The high-level API runs a full pipeline (features, matching, bundle adjustment, exposure compensation, seam finding, blending depending on build flags). `mode=1` (**SCANS**) is often safer than the default panorama mode for mostly planar motion.

In [ ]:
stitcher = cv2.Stitcher_create(cv2.Stitcher_SCANS)
status, stitched = stitcher.stitch([img_a, img_b])

if status == cv2.Stitcher_OK:
    plt.figure(figsize=(16, 8))
    imshow_bgr(stitched, title="cv2.Stitcher on campus crops")
    plt.tight_layout()
    plt.show()
else:
    print("Stitcher failed with status:", status)

## ✏️ Exercises

### Exercise 1: Build your own transformation pipeline

Use `img_a` and create a custom affine transformation experiment.

1. Rotate the image by **90 degrees**.
2. Flip the image **horizontally**.
3. Combine both transformations and compare the result of applying them in different orders.
4. Explain briefly: why are the results for `A @ B` and `B @ A` usually different?

**Hint:** treat transforms as matrix multiplications and compare both the visual output and the resulting transformation matrix.

In [ ]:
# Starter code for Exercise 1
h, w = img_a.shape[:2]
center = (w / 2, h / 2)

# TODO: Rotation by 90 degrees around image center
R90 = np.eye(3)[:2, :]

# TODO: Horizontal flip as affine matrix around image center
Fh = np.eye(3)[:2, :]

# Compose in homogeneous coordinates to compare order
R90_h = np.vstack([R90, [0, 0, 1]]).astype(np.float32)
Fh_h = np.vstack([Fh, [0, 0, 1]]).astype(np.float32)

A_then_B = (Fh_h @ R90_h)[:2, :]
B_then_A = (R90_h @ Fh_h)[:2, :]

im_rot = cv2.warpAffine(img_a, R90, (w, h))
im_flip = cv2.warpAffine(img_a, Fh, (w, h))
im_ab = cv2.warpAffine(img_a, A_then_B, (w, h))
im_ba = cv2.warpAffine(img_a, B_then_A, (w, h))

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(to_rgb(im_rot)); axes[0].set_title("Rotate 90°")
axes[1].imshow(to_rgb(im_flip)); axes[1].set_title("Horizontal flip")
axes[2].imshow(to_rgb(im_ab)); axes[2].set_title("Flip @ Rotate")
axes[3].imshow(to_rgb(im_ba)); axes[3].set_title("Rotate @ Flip")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Are the composed matrices equal?", np.allclose(A_then_B, B_then_A))

### Exercise 2: Least squares vs. RANSAC under different match quality

Use the campus pair and compare model estimates when you change the Lowe ratio threshold.

1. Try at least three ratio values (for example `0.5`, `0.75`, `0.95`).
2. For each setting, compute the translation with:
   - least squares (`fit_translation_ls`)
   - RANSAC (`cv2.estimateAffine2D(..., method=cv2.RANSAC)`)
3. Report and compare:
   - number of retained matches
   - number of RANSAC inliers
   - translation estimate and error to the ground truth
4. Discuss: when does RANSAC clearly outperform least squares?

---

### Exercise 3: Stitch your own smartphone images

Go outside and capture your own data.

1. Take **two smartphone photos** of the same scene with roughly **40-60% overlap** (ideally with mostly planar structure and little moving content).
2. Run the full pipeline from this notebook:
   - feature detection + matching
   - robust homography estimation with RANSAC
   - warping + blending
3. Evaluate your result:
   - Does stitching work directly?
   - If not, what fails (too few matches, low inlier ratio, ghosting, seam artifacts)?
4. Try to improve the result by changing capture conditions (camera motion, overlap, viewpoint) and summarize what worked best.

In [ ]:
# Your code here
